# Día 2 · Construir un panel de siniestros en Barcelona

**Pregunta del panel.** ¿Qué zonas y franjas conviene revisar primero durante un periodo concreto? Vamos a combinar tres gráficos del día 1 con un indicador. El filtro de distrito y meses se aplica una sola vez. Todas las vistas responden a la misma selección.

## 0 · Abrir y preparar

Abre el notebook en Jupyter, VS Code o Colab. En el repositorio, el CSV está en `PySpark/data`. En Colab, súbelo desde **Archivos** con su nombre original. Si faltan bibliotecas, ejecuta `%pip install pandas "seaborn>=0.13" matplotlib`. Los controles interactivos del final usan `ipywidgets` y son opcionales.

## 1 · Recuperar la tabla del día 1

Volvemos a cargar el CSV para que este notebook se pueda abrir por separado. Comprobamos la coherencia y dejamos una fila por expediente. La categoría de distrito desconocido se conserva.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

archivo = '2023_accidents_causa_conductor_gu_bcn_.csv'
inicio = Path.cwd()
carpetas = [inicio, inicio / 'data', inicio.parent / 'data', inicio / 'PySpark' / 'data']
ruta = next((carpeta / archivo for carpeta in carpetas if (carpeta / archivo).is_file()), None)
if ruta is None:
    raise FileNotFoundError(f'Sube {archivo} al entorno o abre el notebook desde el repositorio.')

registros = pd.read_csv(ruta)
registros[['Numero_expedient', 'Nom_districte', 'Hora_dia', 'Descripcio_causa_mediata']].head()

In [ ]:
campos = {
    'Numero_expedient': 'expediente',
    'Nom_districte': 'distrito',
    'Nom_barri': 'barrio',
    'Descripcio_torn': 'turno',
    'NK_Any': 'year',
    'Mes_any': 'month',
    'Dia_mes': 'day',
    'Hora_dia': 'hora',
    'Latitud_WGS84': 'latitud',
    'Longitud_WGS84': 'longitud',
}
tabla = registros[list(campos)].rename(columns=campos).copy()
tabla['expediente'] = tabla['expediente'].str.strip().replace('', pd.NA)
print(f"Filas de origen: {len(tabla):,}")
print(f"Filas sin expediente: {tabla['expediente'].isna().sum():,}")
tabla = tabla.dropna(subset=['expediente'])

# Antes de quedarnos con una fila, comprobamos que las demás dicen lo mismo.
conflictos = tabla.groupby('expediente').nunique(dropna=False).gt(1).any(axis=1)
if conflictos.any():
    raise ValueError(f'Hay {conflictos.sum()} expedientes con datos contradictorios. Revísalos antes de agrupar.')

siniestros = tabla.drop_duplicates(subset='expediente').copy()
siniestros['fecha'] = pd.to_datetime(siniestros[['year', 'month', 'day']])
siniestros['distrito'] = siniestros['distrito'].replace({'Desconegut': 'Desconocido'})
siniestros['barrio'] = siniestros['barrio'].replace({'Desconegut': 'Desconocido'})
siniestros['turno'] = siniestros['turno'].replace({'Matí': 'Mañana', 'Tarda': 'Tarde', 'Nit': 'Noche'})

dias = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
siniestros['dia_semana'] = siniestros['fecha'].dt.dayofweek.map(dict(enumerate(dias)))
siniestros['mes'] = siniestros['fecha'].dt.to_period('M').dt.to_timestamp()

print(f"Expedientes distintos: {len(siniestros):,}")
print(f"Expedientes con distrito desconocido: {siniestros['distrito'].eq('Desconocido').sum():,}")
siniestros[['distrito', 'barrio', 'fecha', 'hora', 'turno']].head()

## 2 · Filtrar antes de calcular

El distrito y el rango de meses definen qué expedientes entran. De esa misma tabla salen:

- El **indicador**, que cuenta los expedientes seleccionados.
- La **línea**, que muestra el recuento mensual dentro del periodo.
- Las **barras**, que comparan distritos cuando eliges todos. Si eliges un distrito, comparan sus barrios.
- El **mapa de calor**, que cruza día de la semana y hora.

Incluimos meses y horas con cero expedientes en la selección. Si no hay ningún expediente, el panel lo indica.

In [ ]:
meses = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio',
         'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
todos = 'Todos los distritos'

def resumir_panel(distrito, primer_mes, ultimo_mes):
    if not 1 <= primer_mes <= ultimo_mes <= 12:
        raise ValueError('Elige un periodo válido entre enero y diciembre.')
    seleccion = siniestros[siniestros['fecha'].dt.month.between(primer_mes, ultimo_mes)]
    zona = 'distrito' if distrito == todos else 'barrio'
    if distrito != todos:
        seleccion = seleccion[seleccion['distrito'] == distrito]

    tendencia = (seleccion.groupby(seleccion['fecha'].dt.month).size()
                 .reindex(range(primer_mes, ultimo_mes + 1), fill_value=0))
    zonas = seleccion[zona].value_counts()
    calor = (pd.crosstab(seleccion['dia_semana'], seleccion['hora'])
             .reindex(index=dias, columns=range(24), fill_value=0))
    total = len(seleccion)
    assert total == tendencia.sum() == zonas.sum() == calor.to_numpy().sum()
    return total, tendencia, zonas, calor, zona

## 3 · Dibujar el panel

Usamos subplots para colocar tres gráficos en una sola figura. El título indica qué estás viendo y el recuento aparece arriba. El mapa de calor conserva el orden de los días y las 24 horas. Al cambiar la selección se recalculan los tres gráficos. La comprobación del paso anterior verifica que todos cuentan el mismo total. La escala de color se adapta a la selección. Para comparar dos paneles, lee los recuentos de sus escalas.

In [ ]:
def mostrar_panel(distrito=todos, primer_mes=1, ultimo_mes=12):
    total, tendencia, zonas, calor, zona = resumir_panel(distrito, primer_mes, ultimo_mes)
    periodo = f'{meses[primer_mes - 1]} a {meses[ultimo_mes - 1]} de 2023'
    if primer_mes == ultimo_mes:
        periodo = f'{meses[primer_mes - 1]} de 2023'
    if total == 0:
        print(f'{distrito} · {periodo} · 0 expedientes registrados')
        return

    fig, axes = plt.subplot_mosaic([['mes', 'zona'], ['hora', 'hora']], figsize=(12, 8))
    fig.suptitle(f'{distrito} · {periodo} · {total:,} expedientes', fontsize=15)

    sns.lineplot(x=tendencia.index, y=tendencia.values, marker='o', ax=axes['mes'])
    axes['mes'].set(title='Evolución mensual', xlabel='Mes', ylabel='Expedientes')
    axes['mes'].set_xticks(tendencia.index, [meses[m - 1][:3] for m in tendencia.index])

    sns.barplot(x=zonas.values, y=zonas.index, color='#613A89', ax=axes['zona'])
    axes['zona'].set(title=f'Recuento por {zona}', xlabel='Expedientes', ylabel='')

    sns.heatmap(calor, cmap='Blues', cbar_kws={'label': 'Expedientes'}, ax=axes['hora'])
    axes['hora'].set(title='Día de la semana y hora', xlabel='Hora del día', ylabel='')
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    plt.show()

mostrar_panel()

## 4 · Cambiar la selección y comprobar el resultado

Primero mira todos los distritos. Después limita la vista a Eixample y al primer semestre. Comprueba qué cambia en el indicador, la línea y el mapa de calor. Las barras ahora muestran barrios. El panel anual mantiene la opción «Desconocido» para revisar qué expedientes carecen de distrito.

In [ ]:
mostrar_panel(distrito='Eixample', primer_mes=1, ultimo_mes=6)

## 5 · Añadir controles interactivos

Los controles llaman a la misma función que acabamos de probar. Si tu entorno admite widgets, podrás elegir un distrito y mover el intervalo de meses sin editar el código. Si falta la biblioteca, instala `ipywidgets` con `%pip install ipywidgets` o sigue usando la función del paso 4.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    print('Puedes usar el panel del paso 4 sin controles interactivos.')
else:
    selector = widgets.Dropdown(
        options=[todos] + sorted(siniestros['distrito'].unique()), description='Distrito',
    )
    rango = widgets.SelectionRangeSlider(
        options=list(zip(meses, range(1, 13))), index=(0, 11),
        description='Meses', continuous_update=False,
    )

    def actualizar(distrito, periodo):
        mostrar_panel(distrito, *periodo)

    salida = widgets.interactive_output(actualizar, {'distrito': selector, 'periodo': rango})
    display(widgets.VBox([selector, rango, salida]))

## 6 · Exportar la tabla que usaremos en las otras versiones

Una fila de la tabla exportada representa una combinación de fecha, distrito, barrio y hora. La columna `expedientes` guarda cuántos siniestros hay en esa combinación. Para recuperar el total hay que **sumar** esa columna. Los notebooks 17 y 18 usarán el mismo archivo.

Ejecuta esta celda para guardar `resumen_siniestros_panel.csv` junto al CSV original. En el repositorio, quedará en `PySpark/data`. En Colab aparecerá en **Archivos**.

In [ ]:
tabla_panel = (siniestros.groupby(['fecha', 'distrito', 'barrio', 'hora'])
               .size().rename('expedientes').reset_index())
assert tabla_panel['expedientes'].sum() == len(siniestros)

destino = ruta.parent / 'resumen_siniestros_panel.csv'
tabla_panel.to_csv(destino, index=False, encoding='utf-8-sig')
print(f'Tabla guardada en {destino.resolve()}')
tabla_panel.head()

## Cierre · Usar el panel para decidir qué revisar

1. Elige un distrito y un periodo. Anota la selección, el total y la franja que destaca.
2. Compara esa selección con otro periodo del mismo distrito. ¿Qué se mantiene? ¿Qué cambia? Si los periodos tienen diferente duración, compara también expedientes por día.
3. Explica qué expedientes revisarías primero y qué información pedirías para comparar el riesgo. El recuento muestra lo registrado. La exposición al tráfico es una de las piezas que faltan.

En el día 3 usaremos esa propuesta para discutir cómo medir si una decisión aporta valor.

[Siguiente: Streamlit y Dash](17_panel_streamlit_y_dash_dia_2.ipynb) · [Día 3: Power BI](18_power_bi_y_valor_dia_3.ipynb) · [Volver al README](../README.md) · [Día 1: galería de visualizaciones](15_visualizacion_siniestros_dia_1.ipynb) · [Controles de Jupyter](https://ipywidgets.readthedocs.io/en/stable/examples/Using%20Interact.html)